# Train Laya for the GATE Block (KYC document agent, Block 1)

This notebook starts from a **fresh** `convaiinnovations/laya-multilingual` checkpoint, trains it on the GATE Block's decision data from Hugging Face, checks it, and publishes it.

1. **Publish the training data** (only the first time): `RADAXGRS/kyc-gate-laya-decisions`, public, with a card that says where every row comes from and what the labels are.
2. **Smoke test** (a few minutes): the whole pipeline on 64 rows, to catch problems before the long run.
3. **Train**, whole model:
   * stage 1, **SFT** (`soft-ce`): Laya learns the job, accept / enhance / reject, the reason and the DocRes task;
   * stage 2, **RL** (`rlcd`): Laya samples its own answers and is rewarded with proper scoring rules, which pay most when the stated probability matches how often the answer is right. This stage trains the *confidence*.
4. **Check** on the held-out split: accuracy, false accepts, calibration (ECE) and the reliability table (of the answers given with >= 95 % confidence, how many were right).
5. **Publish** `RADAXGRS/laya-kyc-gate`: public only if every check passes; otherwise nothing is published.

**Before you run:** Settings -> Accelerator **GPU (T4 or P100)**, Internet **on**, and Add-ons -> Secrets -> add **`HF_TOKEN`** (a Hugging Face token with *write* access) and switch it on for this notebook.

In [ ]:
!pip install -q "laya==0.4.31" "huggingface_hub>=0.30"
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import os

from kaggle_secrets import UserSecretsClient

os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")   # never printed
HF_USER = "RADAXGRS"
BASE = "convaiinnovations/laya-multilingual"                 # the fresh starting point (Apache-2.0)
DATASET = f"{HF_USER}/kyc-gate-laya-decisions"
MODEL = f"{HF_USER}/laya-kyc-gate"

from huggingface_hub import HfApi

print("logged in as", HfApi().whoami()["name"])

In [ ]:
# the training scripts and the data files live in the project repository
!rm -rf /kaggle/working/kyc_zain && git clone -q --depth 1 https://github.com/xgrs8/kyc_zain /kaggle/working/kyc_zain
TRAINING = "/kaggle/working/kyc_zain/main/GATE Block/training"
!ls "{TRAINING}" "{TRAINING}/data/v1"

## 1. Publish the training data (first run only)

Leaves the dataset alone if it already exists (add `--update` to upload a new version).

### Optional: generate the situations again

The published data was made with `python -m gate_block laya-data` (see `data/v1/provenance.json`). To make a new version here instead — e.g. after changing the gate's rules or thresholds — set `REGENERATE = True`: the GATE Block simulates photos of its four templates and draws measurement states, labels each with its rules, and the next cell publishes the result as a new version of the dataset (`--update`). The real photos of v1 are not in the repository, so a regenerated set has only simulated and synthetic rows.

In [ ]:
REGENERATE = False
DATA_DIR = f"{TRAINING}/data/v1"
if REGENERATE:
    GATE = TRAINING.rsplit('/training', 1)[0]
    !pip install -q -r "{GATE}/requirements.txt"
    !cd "{GATE}" && python -m gate_block laya-data --out /kaggle/working/laya_data --simulate 300 --synthetic 3200 --synthetic-enhance 1700 --seed 0
    DATA_DIR = "/kaggle/working/laya_data"

In [ ]:
UPDATE = "--update" if REGENERATE else ""
!python "{TRAINING}/publish_dataset.py" --data "{DATA_DIR}" --stage /kaggle/working/hf_dataset --push {DATASET} {UPDATE}

## 2. Smoke test (a few minutes, nothing is published)

In [ ]:
!python "{TRAINING}/train_laya.py" --base {BASE} --data {DATASET} --out /kaggle/working/smoke --smoke 64 --sft-epochs 1 --rl-epochs 1 --micro-batch 8 --grad-accum 1

## 3-5. Train, check and publish

Stage 1 (SFT, 3 epochs) and stage 2 (RL, 2 epochs) on the whole model; on a T4 expect roughly half an hour to an hour. The run is published publicly only if every gate passes (decision accuracy >= 0.98, no false accept, decision ECE <= 0.05, answers given with >= 95 % confidence right >= 95 % of the time).

In [ ]:
!python "{TRAINING}/train_laya.py" --base {BASE} --data {DATASET} --out /kaggle/working/run --push {MODEL}

In [ ]:
import json

import pandas as pd

report = json.load(open("/kaggle/working/run/report.json"))
ev = report["evaluation"]
print("base:", report["base"])
print("data:", report["data"])
print("accuracy:", ev["accuracy"], "| false accepts:", ev["false_accepts"], "| ECE:", ev["ece"])
print("high confidence:", ev["high_confidence"])
print("gates:", "all passed" if not report["gates_failed"] else report["gates_failed"])
pd.DataFrame(ev["reliability"]["decision"])

## What the confidence means, and phase 2

The labels are the GATE Block's rules, so this model's 95 % means *"agrees with the rules 95 % of the time"*. The KYC system's final confidence is a **formula** (not a model) that combines the OCR's field confidences, the cross-document checks and the gate's scores. For *"95 % means right 95 % of the time"* that formula must be **fitted and then verified on held-out real outcomes** (documents whose OCR result was checked against the A.L.G labels): a reliability table like the one above, per confidence bin, on photos the fit never saw.

Laya can feed that formula a score, e.g. *"will OCR read this photo correctly?"*: once TeleOCR has run on labelled photos, add that question to the data with the real outcome as the answer and run stage 2 (RL with proper-scoring rewards) on it; then fit the formula on one part of the outcomes and check the reliability table on the rest. See `training/README.md` in the repository.